## Introducing the Attention Mechanism

Welcome back! It's amazing seeing you again in our course "Sequence Models & The Dawn of Attention". As we advance to our second lesson, we're about to explore one of the most revolutionary concepts in modern deep learning: the **attention mechanism**.

In our previous lesson, we witnessed firsthand how LSTMs struggle with long-range dependencies, observing performance degradation as sequence lengths increased. This limitation wasn't merely a technical curiosity; it revealed a fundamental bottleneck that restricted the potential of sequence models. The core issue lies in forcing models to compress all relevant information into a **single fixed-size hidden state**, creating an information bottleneck that becomes more severe as sequences grow longer.

Today, we'll discover how attention mechanisms elegantly solve this problem by allowing models to selectively focus on different parts of the input sequence. Rather than relying on a single summary vector, attention enables direct access to any position in the sequence, fundamentally changing how we approach sequence modeling. We'll implement two foundational attention variants:

- **Luong** (multiplicative) attention;
- **Bahdanau** (additive) attention.

We'll look at their mathematical foundations and practical differences. Let's dive in!

---

### The Problem with Fixed Context Windows

The limitation we observed with LSTMs stems from a deeper architectural constraint: the **fixed-size context bottleneck**. Traditional sequence models, like RNNs and LSTMs, process information sequentially and attempt to compress all information encountered so far into a hidden state of a predetermined size. This hidden state must then serve as the sole basis for future predictions or for understanding the sequence as a whole.

Imagine trying to summarize an entire book in a single, short sentence while ensuring all crucial plot points, character developments, and themes are perfectly preserved. It's an incredibly difficult, if not impossible, task. As the book (or sequence) gets longer, more information needs to be crammed into that one sentence (the fixed-size hidden state), inevitably leading to information loss.

Consider a machine translation task. If we want to translate a long, complex sentence like:

> "The cat, which had been lazily napping under the old oak tree in the sprawling garden all afternoon, suddenly awoke with a start,"

an LSTM would process this word by word. By the time it needs to translate "awoke," its hidden state must somehow retain the fact that "cat" is the subject, along with all the intervening descriptive clauses. This reliance on a compressed summary becomes increasingly problematic with longer sequences.

Attention mechanisms offer a way out by allowing the model to **look back at the entire input sequence at each step**, rather than relying solely on a compressed summary.

---

### The Query-Key-Value Paradigm

At its core, attention operates through three fundamental components: **Queries (Q)**, **Keys (K)**, and **Values (V)**. This QKV formulation provides an elegant and powerful framework for computing relevance and retrieving information. The concept is inspired by information retrieval systems, where you use a query to search a database (composed of key-value pairs) to find relevant information.

Let's break down these components in the context of sequence models:

| Component | Role | Intuition |
|-----------|------|-----------|
| **Query** | The current point of interest, i.e. what the model is trying to figure out at a specific step. In machine translation, it might relate to the word currently being generated in the target language. | "Given my current context, what information from the input is most relevant right now?" |
| **Keys** | Associated with different parts of the input sequence. Each key corresponds to a specific element or position in the input. | "Labels" or "indices" for the information in the input. |
| **Values** | Also correspond to the elements of the input sequence. They contain the actual information or content we want to retrieve. For each key there is an associated value. | The content that gets retrieved. |

The attention mechanism works in two steps:

1. **Compare** the Query with all the Keys to calculate a set of scores. These scores determine how relevant each input part (represented by its Key) is to the current Query.
2. **Combine**: use these scores to compute a weighted sum of the Values.

This process allows the model to selectively focus on the most pertinent parts of the input sequence (those whose Keys best match the Query) and retrieve their corresponding Values.

---

### Setting Up Sample Data for Attention

Let's set up some sample data to see what these tensors look like. We'll use PyTorch for our implementation:

```python
import torch

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    torch.manual_seed(42)  # For reproducibility
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)

    return query, keys, values
```

In this function, `create_sample_data`, we generate sample random tensors for our query, keys, and values:

| Tensor | Shape | Meaning |
|--------|-------|---------|
| `query` | `(batch_size, hidden_size)` | A single query vector for each item in the batch. |
| `keys` | `(batch_size, seq_len, hidden_size)` | For each item in the batch, a sequence of `seq_len` key vectors. |
| `values` | `(batch_size, seq_len, hidden_size)` | Mirrors the shape of `keys`; each key has a corresponding value. |

These tensor shapes are typical for attention mechanisms. The `hidden_size` represents the dimensionality of our embeddings or feature vectors.

---

### Calculating Attention Scores

The heart of any attention mechanism lies in computing **attention scores**. These scores quantify the relevance between a Query (representing what we're looking for) and each Key in the input sequence (representing different pieces of information). Think of it like searching for a specific topic (the query) in a library catalog (the keys); some entries will be highly relevant, others not at all.

Once we have these raw scores, they need to be transformed into something more usable. This is where the **softmax** function comes in. Applying softmax to the scores converts them into a set of **attention weights**. These weights have two important properties:

- each weight is between 0 and 1;
- all weights for a given query (across all keys in the sequence) sum up to 1.

This means the attention weights form a probability distribution, indicating how much "attention" or importance the model should pay to each part of the input sequence when constructing its output. A higher weight for a particular key-value pair means that value will contribute more to the final result, known as the **context vector**.

Different attention mechanisms use different scoring functions to calculate the initial raw scores. A common and intuitive one is the **dot product**, which measures the similarity in orientation between the query and key vectors. We'll see this in action with Luong attention.

---

### Implementing Luong Attention

Luong attention, also known as **multiplicative attention**, is a popular and efficient attention mechanism. It gets its "multiplicative" name because it primarily uses dot products (a form of multiplication) to calculate attention scores. Let's see how to implement it.

```python
import torch.nn.functional as F

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism

    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)

    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Ensure query is ready for batch matrix multiplication
    query = query.unsqueeze(1)  # Shape: (batch_size, 1, hidden_size)

    # Compute attention scores using dot product
    # keys.transpose(1, 2) changes keys to (batch_size, hidden_size, seq_len)
    # scores will be (batch_size, 1, seq_len)
    scores = torch.bmm(query, keys.transpose(1, 2))
    scores = scores.squeeze(1)  # Shape: (batch_size, seq_len)

    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)  # Shape: (batch_size, seq_len)

    # Compute context vector as weighted sum of values
    # attention_weights.unsqueeze(1) gives (batch_size, 1, seq_len)
    # values are (batch_size, seq_len, hidden_size)
    # context will be (batch_size, 1, hidden_size)
    context = torch.bmm(attention_weights.unsqueeze(1), values)
    context = context.squeeze(1)  # Shape: (batch_size, hidden_size)

    return context, attention_weights
```

Let's break this down:

1. **Prepare Query:** we first `unsqueeze` the query tensor at dimension 1. This changes its shape from `(batch_size, hidden_size)` to `(batch_size, 1, hidden_size)`, making it compatible for batch matrix multiplication (`torch.bmm`) with the keys.
2. **Calculate Scores:** the core of Luong attention is the dot product between the query and each key. We transpose the last two dimensions of `keys` (from `(batch_size, seq_len, hidden_size)` to `(batch_size, hidden_size, seq_len)`). Then `torch.bmm(query, keys.transpose(1, 2))` computes the dot product of the query with every key in the sequence for each item in the batch. After squeezing out the dimension of size 1, `scores` has shape `(batch_size, seq_len)`, indicating how much the query aligns with each key.
3. **Normalize with Softmax:** the raw scores are passed through `F.softmax(scores, dim=1)`. This operation is applied along `dim=1` (the sequence length dimension), converting the scores into `attention_weights`. These weights now sum to 1 for each item in the batch, representing a probability distribution.
4. **Compute Context Vector:** finally, the context vector is computed as a weighted sum of the values. We `unsqueeze` the `attention_weights` to make them `(batch_size, 1, seq_len)` and then perform another batch matrix multiplication with `values` (shape `(batch_size, seq_len, hidden_size)`). This multiplies each value vector by its attention weight and sums them up. The resulting context vector (after squeezing) has shape `(batch_size, hidden_size)` and summarizes the relevant information from the input sequence based on the query.

Notice how this mechanism enables **parallel processing**: all dot products and weighted sums are computed simultaneously across the sequence and batch, a key advantage over sequential RNN processing.

---

### Implementing Bahdanau Attention

Bahdanau attention, also called **additive attention**, takes a slightly more complex approach to computing query-key similarity. Instead of a simple dot product, it uses a small feed-forward neural network (often a single linear layer after combining query and key) to calculate the scores. This allows for potentially more expressive relationships to be learned.

```python
import torch.nn as nn

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism

    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state (and query/key/value vectors)

    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Define linear layers for Bahdanau attention (these would be part of a model's __init__)
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)

    # Transform query and keys
    query_transformed = W_q(query)  # Shape: (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # Shape: (batch_size, seq_len, hidden_size)

    # Expand query to match keys dimensions for addition
    # query_transformed.unsqueeze(1) -> (batch_size, 1, hidden_size)
    # .expand_as(keys_transformed)   -> (batch_size, seq_len, hidden_size)
    query_expanded = query_transformed.unsqueeze(1).expand_as(keys_transformed)

    # Add transformed query and keys, apply tanh activation
    # This is the "additive" part
    combined_energy = torch.tanh(query_expanded + keys_transformed)  # Shape: (batch_size, seq_len, hidden_size)

    # Compute scores using the 'v' linear layer
    scores = v(combined_energy).squeeze(-1)  # Shape: (batch_size, seq_len)

    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)  # Shape: (batch_size, seq_len)

    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # Shape: (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # Shape: (batch_size, hidden_size)

    return context, attention_weights
```

In Bahdanau attention:

1. **Learnable Transformations:** we define three linear layers: `W_q` to transform the query, `W_k` to transform the keys, and `v` to compute the final score from their combined representation. In a real model, these layers would be initialized once and their weights learned during training.
2. **Transform Query and Keys:** the query and keys are passed through their respective linear layers (`W_q` and `W_k`).
3. **Combine and Activate:** the transformed query is expanded to match the dimensions of the transformed keys. Then they are added together (`query_expanded + keys_transformed`), and the result is passed through a `tanh` activation. This non-linear combination is a key feature of additive attention.
4. **Score Calculation:** the output of `tanh` is passed through the `v` linear layer, which projects it down to a single score per sequence position.
5. **Softmax and Context Vector:** the rest is similar to Luong attention: the scores are normalized with softmax to get `attention_weights`, and these weights are used to compute a weighted sum of the values to get the context vector.

> **Key difference:** Luong uses a direct dot product to compute scores, while Bahdanau uses a small neural network with learned weights, allowing for more complex scoring functions.

---

### Comparing Attention Mechanisms

Now let's examine how these two attention mechanisms behave in practice by running them on our sample data and analyzing their outputs:

```python
import numpy as np

def main():
    # Create sample data
    batch_size = 2
    seq_len = 6
    hidden_size = 4
    query, keys, values = create_sample_data(batch_size, seq_len, hidden_size)

    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")

    # Apply Luong attention
    context_luong, attention_weights_luong = luong_attention(query, keys, values)

    print(f"Context vector shape: {context_luong.shape}")  # Luong context
    print(f"Attention weights shape: {attention_weights_luong.shape}")  # Luong weights

    # Display Luong attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights_luong.shape[0]):  # Iterate through batch
        weights = attention_weights_luong[i].detach().numpy()
        print(f"Batch {i}: {np.round(weights, decimals=8)}")  # Using numpy for cleaner print

    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=hidden_size)

    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):  # Iterate through batch
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {np.round(weights, decimals=8)}")
```

When we run the `main()` function, we'll see the shapes of our tensors and the attention weights produced by each mechanism. The expected output is:

```text
Query shape: torch.Size([2, 4])
Keys shape: torch.Size([2, 6, 4])
Values shape: torch.Size([2, 6, 4])
Context vector shape: torch.Size([2, 4])
Attention weights shape: torch.Size([2, 6])

Luong Attention weights:
Batch 0: [0.02072945 0.7967359  0.07351931 0.09071727 0.01487425 0.00342377]
Batch 1: [0.01174377 0.01791002 0.00222191 0.00307018 0.07629474 0.8887594 ]

Bahdanau Attention weights:
Batch 0: [0.14973381 0.21063526 0.1710429  0.18545005 0.14640713 0.1367309 ]
Batch 1: [0.11768297 0.15604942 0.18164434 0.24428545 0.1681909  0.13214697]
```

Looking at the output:

- **Shapes:** they confirm our understanding. The context vector has the same `hidden_size` as the query, and attention weights are distributed over `seq_len`.
- **Luong attention** tends to produce more "peaked" or concentrated weights. For Batch 0, the second element (index 1) gets almost 80% of the attention. For Batch 1, the last element (index 5) gets about 89%. This is characteristic of dot-product attention, which can focus sharply if a key is very similar to the query.
- **Bahdanau attention**, with its learnable layers and `tanh` activation, often produces more "diffuse" or spread-out weights, especially before training. In our example (with randomly initialized weights for `W_q`, `W_k`, `v`), the weights are more evenly distributed. For Batch 0, the highest weight is around 21%, and for Batch 1, around 24%. This doesn't mean it can't focus; during training, it can learn to produce sharper distributions if needed.

The choice between Luong and Bahdanau (or other attention variants) can depend on the specific task, dataset, and computational budget. Luong is generally faster due to its simpler computation, while Bahdanau offers more flexibility through its learnable components.

---

### Conclusion and Next Steps

Today, we've taken a crucial step in our journey from traditional sequence models to the Transformer architecture. We explored how attention mechanisms solve the fixed-context bottleneck that limited RNNs and LSTMs, introducing the elegant Query-Key-Value paradigm that enables selective information retrieval from any sequence position.

Through implementing both Luong and Bahdanau attention, we discovered how different similarity functions produce distinct attention behaviors: multiplicative attention can create focused patterns, while additive attention with learnable transformations allows for more nuanced distributions. These mechanisms are foundational building blocks for more advanced attention systems.

In our next lesson, we'll build upon these concepts to explore multi-head attention, discovering how parallel attention mechanisms can capture different types of relationships simultaneously, bringing us closer to the full Transformer. Until then, let's practice what we've learned today!

## Building Your First QKV Tensors

After learning about Query, Key, and Value vectors in the lesson, you'll now get hands-on experience creating these fundamental building blocks of attention. Your goal is to implement the `create_sample_data` function, which generates sample QKV tensors for testing our attention mechanisms.

---

### Your Task

The provided code includes complete implementations of both Luong and Bahdanau attention, but `create_sample_data` needs your help. You'll need to:

- set the random seed for reproducible results;
- create the `keys` and `values` tensors with the correct dimensions;
- create a `query` tensor;
- return all three tensors.

Pay close attention to the tensor shapes: getting the dimensions right is crucial for the attention mechanisms to work properly. Once you complete this function, you'll see both attention types in action!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)
    
    # Expand query to match keys dimensions
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)
    
    # Apply tanh activation and compute scores
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # TODO: Implement this function to create sample QKV tensors
    # 1. Set random seed to 42 for reproducibility
    # 2. Create keys tensor with shape (batch_size, seq_len, hidden_size)
    # 3. Create values tensor with shape (batch_size, seq_len, hidden_size)
    # 4. Create query tensor with shape (batch_size, hidden_size)
    # 5. Return query, keys, values in that order
    pass


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()

```

---

### Solution

Only `create_sample_data` was missing:

- `torch.manual_seed(42)` is called first, so the random data is reproducible.
- `keys` and `values` have shape `(batch_size, seq_len, hidden_size)`: one vector per sequence position, and each key has a matching value.
- `query` has shape `(batch_size, hidden_size)`: one query vector per batch item.
- The tensors are created in the order keys, values, query, and returned as `query, keys, values`. The creation order matters, because it decides which random numbers each tensor gets.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)
    
    # Expand query to match keys dimensions
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)
    
    # Apply tanh activation and compute scores
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    torch.manual_seed(42)  # For reproducibility
    keys = torch.randn(batch_size, seq_len, hidden_size)    # (batch, seq_len, hidden)
    values = torch.randn(batch_size, seq_len, hidden_size)  # (batch, seq_len, hidden)
    query = torch.randn(batch_size, hidden_size)            # (batch, hidden)

    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()


## Building Attention Score Engine

Now that you've mastered creating Query, Key, and Value tensors, it's time to dive into the heart of attention: calculating attention scores using the Luong mechanism.

---

### Your Task

In this exercise, you'll complete the core scoring step of Luong attention. The provided code has most of the `luong_attention` function ready, but two important lines are missing from the score calculation section. You need to:

- calculate the dot-product scores between the prepared query and the transposed keys using batch matrix multiplication (`torch.bmm`);
- remove the extra dimension from the resulting `scores` tensor with `squeeze`.

The goal is to understand how Luong attention computes similarity scores through matrix operations. Pay attention to the tensor shapes and how the transpose operation allows the dot product to work correctly across batches. Once you fill in these lines, you'll see how the attention mechanism decides which parts of the input sequence to focus on!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    # TODO: Calculate dot-product scores between query and keys using torch.bmm
    # TODO: Remove the unnecessary dimension from scores using squeeze
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)
    
    # Expand query to match keys dimensions
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)
    
    # Apply tanh activation and compute scores
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()

```

---

### Solution

Only the score calculation in `luong_attention` was missing:

- `keys.transpose(1, 2)` turns the keys from `(batch_size, seq_len, hidden_size)` into `(batch_size, hidden_size, seq_len)`, so the shapes line up for a matrix product.
- `torch.bmm(query, keys.transpose(1, 2))` multiplies `(batch_size, 1, hidden_size)` by `(batch_size, hidden_size, seq_len)`. The result is `(batch_size, 1, seq_len)`: the dot product of the query with every key.
- `scores.squeeze(1)` removes the dimension of size 1, giving `(batch_size, seq_len)`, which is what `F.softmax(scores, dim=1)` expects.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    # keys.transpose(1, 2): (batch_size, hidden_size, seq_len)
    # bmm: (batch_size, 1, hidden_size) x (batch_size, hidden_size, seq_len) -> (batch_size, 1, seq_len)
    scores = torch.bmm(query, keys.transpose(1, 2))
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)
    
    # Expand query to match keys dimensions
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)
    
    # Apply tanh activation and compute scores
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()


## From Scores to Context Vector

Now that you've successfully calculated attention scores, it's time to complete the Luong attention mechanism by converting those raw scores into the final context vector.

---

### Your Task

In the provided code, the `luong_attention` function already has the score calculation working from your previous work. You need to complete the remaining steps to turn those scores into a meaningful context representation:

- apply **softmax** normalization to convert scores into proper attention weights;
- compute the weighted **context vector** using the attention weights and values (`torch.bmm`);
- ensure the output has the correct dimensions (remove the extra dimension with `squeeze`).

The rest of the code is ready to run once you finish these steps. This will give you a complete, working attention system to experiment with!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # TODO: Apply softmax to get attention weights
    
    # TODO: Compute context vector using batch matrix multiplication between attention_weights and values
    
    # TODO: Remove the unnecessary dimension from context
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)
    
    # Expand query to match keys dimensions
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)
    
    # Apply tanh activation and compute scores
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()

```

---

### Solution

Three steps were missing at the end of `luong_attention`:

- `F.softmax(scores, dim=1)` normalizes the scores over the sequence dimension, so each row of `attention_weights` is between 0 and 1 and sums to 1.
- `torch.bmm(attention_weights.unsqueeze(1), values)` multiplies `(batch_size, 1, seq_len)` by `(batch_size, seq_len, hidden_size)`. This is the weighted sum of the value vectors, with shape `(batch_size, 1, hidden_size)`.
- `context.squeeze(1)` removes the dimension of size 1, giving the final context vector of shape `(batch_size, hidden_size)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax over the sequence dimension to get attention weights
    attention_weights = F.softmax(scores, dim=1)  # (batch_size, seq_len)

    # Weighted sum of values: (batch_size, 1, seq_len) x (batch_size, seq_len, hidden_size)
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)

    # Remove the extra dimension
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)
    
    # Expand query to match keys dimensions
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)
    
    # Apply tanh activation and compute scores
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()


## Building Complex Attention Mechanisms

Now that you've successfully implemented the complete Luong attention mechanism, it's time to explore a different approach to attention scoring. In this exercise, you'll implement the key components of **Bahdanau attention**, which uses an additive approach instead of the multiplicative dot product you just worked with.

---

### Your Task

Your goal is to complete the missing parts in the `bahdanau_attention` function. The provided code already includes the linear layer definitions and the final scoring steps, but you need to implement the core transformation and combination steps:

- transform the query and keys using the provided linear layers `W_q` and `W_k`;
- expand the transformed query to match the keys' dimensions for element-wise addition;
- combine the transformed representations and apply the `tanh` activation.

This additive approach is what makes Bahdanau attention unique and sometimes more flexible than simple dot-product attention.

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # TODO: Transform the query using W_q to get query_transformed
    
    # TODO: Transform the keys using W_k to get keys_transformed
    
    # TODO: Expand query_transformed to match the shape of keys_transformed
    # Hint: Use unsqueeze(1) and expand(-1, seq_len, -1)
    
    # TODO: Calculate combined by adding the expanded query and keys_transformed, 
    # then apply torch.tanh activation
    
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()

```

---

### Solution

Four lines were missing in `bahdanau_attention`:

- `W_q(query)` projects the query to `(batch_size, hidden_size)`, and `W_k(keys)` projects every key to `(batch_size, seq_len, hidden_size)`.
- `query_transformed.unsqueeze(1).expand(-1, seq_len, -1)` adds a sequence dimension and repeats the query across it, so its shape matches `keys_transformed`.
- `torch.tanh(query_expanded + keys_transformed)` adds the two (the "additive" part) and applies a non-linearity. The existing `v(combined).squeeze(-1)` then turns this into one score per position.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys with the learned linear layers
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)

    # Expand the query to match the keys' shape for element-wise addition
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)

    # Additive combination followed by tanh
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)

    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()


## Finishing Bahdanau Attention

After building the linear transformations and energy combination for Bahdanau attention in the previous exercise, you're now ready to complete the full attention mechanism. Your goal is to finish the `bahdanau_attention` function by implementing the final scoring, normalization, and context computation steps.

---

### Your Task

The provided code already handles the query and key transformations, expansion, and `tanh` activation to create the combined energy. Now, you need to complete the mechanism by adding three key steps:

- calculate the final **scores** by passing the combined energy through the `v` linear layer;
- apply **softmax** to normalize these scores into attention weights;
- compute the **context vector** as the weighted sum of values using these weights.

These final steps are similar to what you've seen in Luong attention, connecting the concepts you've learned. Complete these steps, and you'll have both major attention mechanisms working together!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys with the learned linear layers
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)

    # Expand the query to match the keys' shape for element-wise addition
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)

    # Additive combination followed by tanh
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)

    # TODO: Calculate scores by passing combined through the v linear layer and removing the last dimension

    # TODO: Apply softmax over the sequence dimension to get attention_weights

    # TODO: Compute context as the weighted sum of values using torch.bmm, then remove the extra dimension

    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()

```

---

### Solution

Three steps were missing at the end of `bahdanau_attention`:

- `v(combined).squeeze(-1)` projects the combined energy from `(batch_size, seq_len, hidden_size)` down to one score per position, `(batch_size, seq_len)`.
- `F.softmax(scores, dim=1)` normalizes the scores over the sequence dimension, so the attention weights are between 0 and 1 and sum to 1.
- `torch.bmm(attention_weights.unsqueeze(1), values)` computes the weighted sum of the values, and `squeeze(1)` removes the extra dimension to give `(batch_size, hidden_size)`.

These last steps are identical to the Luong version. The only difference between the two mechanisms is how the scores are computed: a dot product for Luong, a small learned network for Bahdanau.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def luong_attention(query, keys, values):
    """
    Implement Luong (multiplicative) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    # Calculate attention scores using dot product
    query = query.unsqueeze(1)  # (batch_size, 1, hidden_size)
    
    # Compute attention scores
    scores = torch.bmm(query, keys.transpose(1, 2))  # (batch_size, 1, seq_len)
    scores = scores.squeeze(1)  # (batch_size, seq_len)
    
    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)
    
    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)
    
    return context, attention_weights

def bahdanau_attention(query, keys, values, hidden_size):
    """
    Implement Bahdanau (additive) attention mechanism
    
    Args:
        query: (batch_size, hidden_size)
        keys: (batch_size, seq_len, hidden_size)
        values: (batch_size, seq_len, hidden_size)
        hidden_size: dimension of hidden state
    
    Returns:
        context: (batch_size, hidden_size)
        attention_weights: (batch_size, seq_len)
    """
    batch_size, seq_len, _ = keys.shape
    
    # Define linear layers for Bahdanau attention
    W_q = nn.Linear(hidden_size, hidden_size, bias=False)
    W_k = nn.Linear(hidden_size, hidden_size, bias=False)
    v = nn.Linear(hidden_size, 1, bias=False)
    
    # Transform query and keys with the learned linear layers
    query_transformed = W_q(query)  # (batch_size, hidden_size)
    keys_transformed = W_k(keys)    # (batch_size, seq_len, hidden_size)

    # Expand the query to match the keys' shape for element-wise addition
    query_expanded = query_transformed.unsqueeze(1).expand(-1, seq_len, -1)  # (batch_size, seq_len, hidden_size)

    # Additive combination followed by tanh
    combined = torch.tanh(query_expanded + keys_transformed)  # (batch_size, seq_len, hidden_size)

    # Calculate scores by passing the combined energy through v
    scores = v(combined).squeeze(-1)  # (batch_size, seq_len)

    # Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=1)  # (batch_size, seq_len)

    # Compute context vector as weighted sum of values
    context = torch.bmm(attention_weights.unsqueeze(1), values)  # (batch_size, 1, hidden_size)
    context = context.squeeze(1)  # (batch_size, hidden_size)

    return context, attention_weights

def create_sample_data(batch_size=2, seq_len=6, hidden_size=4):
    """Create sample query, keys, and values for demonstration"""
    # Create sample data
    torch.manual_seed(42)
    keys = torch.randn(batch_size, seq_len, hidden_size)
    values = torch.randn(batch_size, seq_len, hidden_size)
    query = torch.randn(batch_size, hidden_size)
    
    return query, keys, values


def main():   
    # Create sample data
    query, keys, values = create_sample_data()
    
    print(f"Query shape: {query.shape}")
    print(f"Keys shape: {keys.shape}")
    print(f"Values shape: {values.shape}")
    
    # Apply Luong attention
    context, attention_weights = luong_attention(query, keys, values)
    
    print(f"Context vector shape: {context.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")
    
    # Display attention weights
    print("\nLuong Attention weights:")
    for i in range(attention_weights.shape[0]):
        weights = attention_weights[i].detach().numpy()
        print(f"Batch {i}: {weights}")    
    
    # Apply Bahdanau attention
    context_bahdanau, attention_weights_bahdanau = bahdanau_attention(query, keys, values, hidden_size=4)
    
    # Display Bahdanau attention weights
    print("\nBahdanau Attention weights:")
    for i in range(attention_weights_bahdanau.shape[0]):
        weights = attention_weights_bahdanau[i].detach().numpy()
        print(f"Batch {i}: {weights}")


if __name__ == "__main__":
    main()
